### Homework 2
## Appied Machine Learning

#### Harsh Agrawal (ha455)
#### Daniel Xing (yx753)

In [1]:
## Q1) b) Split the Training Dataset

import pandas as pd
from sklearn.model_selection import train_test_split

# Reading the training dataset
df = pd.read_csv("train.csv")

# Split into 70% training and 30% development sets
train_set, dev_set = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["target"]
)

# Save both datasets
train_set.to_csv("malt_train_data.csv", index=False)
dev_set.to_csv("malt_dev_data.csv", index=False)

# Display dataset sizes
print("Original dataset:", len(df))
print("Training set:", len(train_set))
print("Development set:", len(dev_set))




Original dataset: 5026
Training set: 3518
Development set: 1508


In [2]:
## Q1) c) Preprocess the Data

import re
import pandas as pd

def preprocess_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize case
    text = text.lower()

    # Replace long hexadecimal hashes with a placeholder
    text = re.sub(
        r'\b[a-f0-9]{32,}\b',
        ' HASH ',
        text
    )

    # Normalize repeated whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Reading the Reading datasets
train_df = pd.read_csv("malt_train_data.csv")
dev_df = pd.read_csv("malt_dev_data.csv")

# Apply identical preprocessing
train_df["text"] = train_df["text"].apply(preprocess_text)
dev_df["text"] = dev_df["text"].apply(preprocess_text)

# Save the preprocessed datasets
train_df.to_csv("malt_train_preprocessed.csv", index=False)
dev_df.to_csv("malt_dev_preprocessed.csv", index=False)

print("Preprocessing complete!")
print("Training rows:", len(train_df))
print("Development rows:", len(dev_df))


Preprocessing complete!
Training rows: 3518
Development rows: 1508


In [3]:
## Q1) d) Bag of words model

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Reading the Reading training and development splits
train_df = pd.read_csv("malt_train_data.csv")
dev_df = pd.read_csv("malt_dev_data.csv")

# Choose minimum document frequency
M = 5

# Fit the vectorizer ONLY on the training text
vectorizer = CountVectorizer(
    binary=True,
    min_df=M
)

X_train = vectorizer.fit_transform(train_df["text"].fillna(""))
X_dev = vectorizer.transform(dev_df["text"].fillna(""))

# Report dimensions
print("Threshold M:", M)
print("Training feature matrix:", X_train.shape)
print("Development feature matrix:", X_dev.shape)
print("Total number of features:", len(vectorizer.vocabulary_))

# Verify that features are binary
print("Training feature values:", set(X_train.data))


Threshold M: 5
Training feature matrix: (3518, 6053)
Development feature matrix: (1508, 6053)
Total number of features: 6053
Training feature values: {np.int64(1)}


In [4]:
## Q1) e) i) Logistic Regression Model

import pandas as pd
import numpy as np
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Reading the Reading datasets
train_df = pd.read_csv("malt_train_data.csv")
dev_df = pd.read_csv("malt_dev_data.csv")

# Binary bag-of-words features (M = 5)
vectorizer = CountVectorizer(binary=True, min_df=5)

X_train = vectorizer.fit_transform(train_df["text"].fillna(""))
X_dev = vectorizer.transform(dev_df["text"].fillna(""))

y_train = train_df["target"]
y_dev = dev_df["target"]

# Logistic regression without regularization
model = LogisticRegression(
    penalty=None,
    max_iter=2000,
    solver="lbfgs"
)

model.fit(X_train, y_train)

# Make predictions
train_predictions = model.predict(X_train)
dev_predictions = model.predict(X_dev)

# Calculate F1 scores
train_f1 = f1_score(y_train, train_predictions, zero_division=0)
dev_f1 = f1_score(y_dev, dev_predictions, zero_division=0)

print("Training F1:", round(train_f1, 4))
print("Development F1:", round(dev_f1, 4))
print("Number of features:", X_train.shape[1])


Training F1: 1.0
Development F1: 0.8597
Number of features: 6053


c:\Users\Asus\anaconda3\envs\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [5]:
## Q1) e) ii) Logistic Regression Model

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Reading Reading datasets
train_df = pd.read_csv("malt_train_data.csv")
dev_df = pd.read_csv("malt_dev_data.csv")

# Binary bag-of-words features (M = 5)
vectorizer = CountVectorizer(binary=True, min_df=5)
X_train = vectorizer.fit_transform(train_df["text"].fillna(""))
X_dev = vectorizer.transform(dev_df["text"].fillna(""))

y_train = train_df["target"]
y_dev = dev_df["target"]

# L1-regularized logistic regression
# C = 1 selected using development-set performance
model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    l1_ratio=1,
    max_iter=3000,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluate
train_pred = model.predict(X_train)
dev_pred = model.predict(X_dev)

print("C =", 1.0)
print("Training F1:", round(f1_score(y_train, train_pred), 4))
print("Development F1:", round(f1_score(y_dev, dev_pred), 4))



C = 1.0
Training F1: 0.9891
Development F1: 0.8688


In [6]:
## Q1) e) iii) Logistic Regression Model

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Reading the Reading datasets
train_df = pd.read_csv("malt_train_data.csv")
dev_df = pd.read_csv("malt_dev_data.csv")

# Create binary bag-of-words features (M = 5)
vectorizer = CountVectorizer(binary=True, min_df=5)
X_train = vectorizer.fit_transform(train_df["text"].fillna(""))
X_dev = vectorizer.transform(dev_df["text"].fillna(""))

y_train = train_df["target"]
y_dev = dev_df["target"]

# L2-regularized logistic regression
C = 1.0

model = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    C=C,
    max_iter=3000,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
train_pred = model.predict(X_train)
dev_pred = model.predict(X_dev)

# F1 scores
print("C =", C)
print("Training F1:", round(f1_score(y_train, train_pred), 4))
print("Development F1:", round(f1_score(y_dev, dev_pred), 4))


C = 1.0
Training F1: 1.0
Development F1: 0.8824


c:\Users\Asus\anaconda3\envs\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [7]:
## Q1) e) v) Logistic Regression Model

import numpy as np
import pandas as pd

# Extract feature names and L1 model coefficients
feature_names = vectorizer.get_feature_names_out()
weights = model.coef_[0]

# Largest positive weights: push toward flagged (target = 1)
positive_idx = np.argsort(weights)[-10:][::-1]

# Most negative weights: push toward normal (target = 0)
negative_idx = np.argsort(weights)[:10]

results = pd.DataFrame({
    "Words pushing toward flagged": feature_names[positive_idx],
    "Positive weight": weights[positive_idx],
    "Words pushing toward normal": feature_names[negative_idx],
    "Negative weight": weights[negative_idx]
})

print(results.to_string(index=False))


Words pushing toward flagged  Positive weight Words pushing toward normal  Negative weight
                    sabotage         1.405380                       token        -1.532429
                        time         1.365915                      tokens        -0.839721
                    thinking         1.160799                       print        -0.738544
                        want         1.004234                 performance        -0.563336
               cybersecurity         0.795738                   apologize        -0.563194
                      weaker         0.668339                 experiments        -0.562923
                      subtle         0.665503                     ratings        -0.543584
                instructions         0.648310                         gpt        -0.528904
                         not         0.639018                         def        -0.498166
                       limit         0.626450                         the        -0.484670

In [8]:
## Q1) f) Bernoulli Naive Bayes Model

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# 1. Reading the Reading datasets
train_df = pd.read_csv("malt_train_data.csv")
dev_df = pd.read_csv("malt_dev_data.csv")

y_train = train_df["target"].to_numpy(dtype=int)
y_dev = dev_df["target"].to_numpy(dtype=int)

# 2. Build binary bag-of-words features (M = 5)
vectorizer = CountVectorizer(binary=True, min_df=5)

X_train = vectorizer.fit_transform(
    train_df["text"].fillna("")
)
X_dev = vectorizer.transform(
    dev_df["text"].fillna("")
)

# 3. Train Bernoulli Naive Bayes from scratch
alpha = 1.0
classes = [0, 1]

n_features = X_train.shape[1]
log_priors = np.zeros(2)
log_prob_present = np.zeros((2, n_features))
log_prob_absent = np.zeros((2, n_features))

for k in classes:
    Xk = X_train[y_train == k]
    nk = Xk.shape[0]

    # Number of class-k documents containing each word
    word_counts = np.asarray(Xk.sum(axis=0)).ravel()

    # Laplace-smoothed probabilities
    prob_present = (word_counts + alpha) / (nk + 2 * alpha)
    prob_absent = 1 - prob_present

    # Store log probabilities
    log_priors[k] = np.log(nk / len(y_train))
    log_prob_present[k] = np.log(prob_present)
    log_prob_absent[k] = np.log(prob_absent)

# 4. Predict development labels using log probabilities
def predict(X):
    scores = np.zeros((X.shape[0], 2))

    for k in classes:
        scores[:, k] = (
            X @ (log_prob_present[k] - log_prob_absent[k])
            + log_prob_absent[k].sum()
            + log_priors[k]
        ).A1 if hasattr(
            X @ (log_prob_present[k] - log_prob_absent[k]),
            "A1"
        ) else (
            np.asarray(
                X @ (log_prob_present[k] - log_prob_absent[k])
            ).ravel()
            + log_prob_absent[k].sum()
            + log_priors[k]
        )

    return np.argmax(scores, axis=1)

y_pred = predict(X_dev)

# 5. Calculate F1-score manually
tp = np.sum((y_dev == 1) & (y_pred == 1))
fp = np.sum((y_dev == 0) & (y_pred == 1))
fn = np.sum((y_dev == 1) & (y_pred == 0))

precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0.0
)

print("Number of features:", n_features)
print("True positives:", tp)
print("False positives:", fp)
print("False negatives:", fn)
print("Development F1-score:", round(f1, 4))


Number of features: 6053
True positives: 226
False positives: 202
False negatives: 12
Development F1-score: 0.6787


In [9]:
##Q1) h) N-gram model

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# --------------------------------------------------
# 1. Load the fixed datasets
# --------------------------------------------------
train_df = pd.read_csv("malt_train_data.csv")
dev_df = pd.read_csv("malt_dev_data.csv")

y_train = train_df["target"].to_numpy(dtype=int)
y_dev = dev_df["target"].to_numpy(dtype=int)

# --------------------------------------------------
# 2. Build binary bigram features
# --------------------------------------------------
M = 10

vectorizer = CountVectorizer(
    binary=True,
    min_df=M,
    ngram_range=(2, 2)
)

# Fit vocabulary ONLY on training data
X_train = vectorizer.fit_transform(
    train_df["text"].fillna("")
)

# Use the same vocabulary for development data
X_dev = vectorizer.transform(
    dev_df["text"].fillna("")
)

feature_names = vectorizer.get_feature_names_out()

print("Minimum document frequency M:", M)
print("Training feature matrix:", X_train.shape)
print("Development feature matrix:", X_dev.shape)
print("Total number of 2-grams in the vocabulary:", len(feature_names))

print("\n10 example 2-grams:")
for bigram in feature_names[:10]:
    print(bigram)

# --------------------------------------------------
# 3. Logistic regression with L2 regularization
# --------------------------------------------------
logistic = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=3000,
    random_state=42
)

logistic.fit(X_train, y_train)

train_pred_lr = logistic.predict(X_train)
dev_pred_lr = logistic.predict(X_dev)

print("\n--- Logistic Regression (L2) ---")
print("Training F1:",
      round(f1_score(y_train, train_pred_lr, zero_division=0), 4))
print("Development F1:",
      round(f1_score(y_dev, dev_pred_lr, zero_division=0), 4))

# --------------------------------------------------
# 4. Bernoulli Naive Bayes from scratch
# --------------------------------------------------
alpha = 1.0
n_features = X_train.shape[1]
classes = [0, 1]

log_prior = np.zeros(2)
log_present = np.zeros((2, n_features))
log_absent = np.zeros((2, n_features))

for k in classes:
    Xk = X_train[y_train == k]
    nk = Xk.shape[0]

    # Count documents of class k containing each bigram
    counts = np.asarray(Xk.sum(axis=0)).ravel()

    # Laplace smoothing
    prob_present = (counts + alpha) / (nk + 2 * alpha)
    prob_absent = 1.0 - prob_present

    log_prior[k] = np.log(nk / len(y_train))
    log_present[k] = np.log(prob_present)
    log_absent[k] = np.log(prob_absent)

def predict_bernoulli_nb(X):
    scores = np.zeros((X.shape[0], 2))

    for k in classes:
        scores[:, k] = (
            np.asarray(
                X @ (log_present[k] - log_absent[k])
            ).ravel()
            + log_absent[k].sum()
            + log_prior[k]
        )

    return np.argmax(scores, axis=1)

train_pred_nb = predict_bernoulli_nb(X_train)
dev_pred_nb = predict_bernoulli_nb(X_dev)

print("\n--- Bernoulli Naive Bayes ---")
print("Training F1:",
      round(f1_score(y_train, train_pred_nb, zero_division=0), 4))
print("Development F1:",
      round(f1_score(y_dev, dev_pred_nb, zero_division=0), 4))


Minimum document frequency M: 10
Training feature matrix: (3518, 17777)
Development feature matrix: (1508, 17777)
Total number of 2-grams in the vocabulary: 17777

10 example 2-grams:
00 00
00 movb
000 000
000 length
000 token
000 tokens
000 training
01 71
0123456789 abcdefghijklmnopqrstuvwxyz
0125 max_tokens

--- Logistic Regression (L2) ---
Training F1: 1.0
Development F1: 0.8759

--- Bernoulli Naive Bayes ---
Training F1: 0.6692
Development F1: 0.6438


c:\Users\Asus\anaconda3\envs\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [10]:

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Load all labeled training data and the test data
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
test_labels = pd.read_csv("test_labels.csv")

# Fit the vocabulary using all labeled training transcripts
vectorizer = CountVectorizer(
    binary=True,
    min_df=5
)

X_train = vectorizer.fit_transform(
    train_df["text"].fillna("")
)
X_test = vectorizer.transform(
    test_df["text"].fillna("")
)

y_train = train_df["target"].to_numpy()

# Train the selected classifier
model = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=3000,
    random_state=42
)

model.fit(X_train, y_train)

# Predict test labels
test_predictions = model.predict(X_test)

# Align true labels by ID for evaluation
results = test_df[["id"]].merge(
    test_labels[["id", "target"]],
    on="id",
    how="inner",
    validate="one_to_one"
)

assert len(results) == len(test_df)

# Ensure label ordering matches the test predictions
aligned_test = test_df[["id"]].merge(
    test_labels[["id", "target"]],
    on="id",
    how="left",
    validate="one_to_one"
)

y_test = aligned_test["target"].to_numpy()

test_f1 = f1_score(
    y_test,
    test_predictions,
    zero_division=0
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])
print("Number of features:", X_train.shape[1])
print("Test F1-score:", round(test_f1, 4))


Training samples: 5026
Test samples: 2153
Number of features: 6999
Test F1-score: 0.8837
